# Titanic ① 単純なルールで提出する

機械学習はまだ使わない．データを眺めて決めたルールで予測し，提出ファイルを作って提出するところまでを一通りやる．

ここで出たスコアは，あとでモデルを作ったときの比較基準にする．

## データの準備

データの取得と読み込み，提出ファイルの書き出しは `src/titanic/data.py` にまとめてある．`data/` にデータがなければ，Kaggle API でダウンロードしてから読み込む（Kaggle の API トークンと，コンペへの参加が必要）．

In [ ]:
# src/ 以下を編集したら，カーネルを再起動しなくても読み込み直す
%load_ext autoreload
%autoreload 2

import pandas as pd

from titanic.data import DATA_DIR, load_data, save_submission

# print で DataFrame を出すときに，列が「...」で省略されないようにする
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

train, test = load_data()
print("train:", train.shape, " test:", test.shape)

## データをざっと見る

In [ ]:
print(train.head())

In [ ]:
# 列ごとの欠損数．test 側に欠損があると予測のときに困るので，両方見ておく
# （Survived は test に無いので，test 列は <NA> になる）
print(pd.concat({"train": train.isna().sum(), "test": test.isna().sum()}, axis=1).astype("Int64"))

## 生存率を属性ごとに比べる

評価指標は Accuracy なので，多い方のクラス（死亡）を全員に当てはめるだけでも，死亡者の割合のぶんは正解になる．まずこの最低ラインを確認しておく．

In [ ]:
print(f"全体の生存率: {train['Survived'].mean():.3f}")
print(f"全員「死亡」と予測したときの train での正解率: {(train['Survived'] == 0).mean():.3f}")

In [ ]:
# Survived は 0/1 なので，平均をとればそのまま生存率になる
print(train.groupby("Sex")["Survived"].agg(["mean", "count"]))
print()
print(train.groupby("Pclass")["Survived"].agg(["mean", "count"]))

In [ ]:
# 性別 × 等級の生存率
print(train.pivot_table(index="Sex", columns="Pclass", values="Survived", aggfunc="mean").round(3))

性別による差がいちばん大きい．救命ボートには「女性と子どもが先」という方針で乗せていたので，データにもそれが出ている．

そこで「女性は生存，男性は死亡」というルールで予測する．Data タブにある見本 `gender_submission.csv` と同じ予測になる．

一方で，性別×等級の表を見ると 3 等の女性はほかの等級ほど助かっていない．性別だけのルールではこういう差を拾えないので，次の実験でモデルに任せる．

In [ ]:
def predict_by_sex(df: pd.DataFrame) -> pd.Series:
    """女性なら 1（生存），男性なら 0（死亡）を返す．"""
    return (df["Sex"] == "female").astype(int)


train_acc = (predict_by_sex(train) == train["Survived"]).mean()
print(f"性別ルールの train での正解率: {train_acc:.3f}")

train を見て決めたルールなので，test でのスコアはこれより少し下がるかもしれない．実際のところは提出して確かめる．

## 提出ファイルを作る

In [ ]:
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": predict_by_sex(test),
})

# 形式のチェックもここでやる（中身は src/titanic/data.py）
path = save_submission(submission, "01_rule_baseline")
print("書き出し先:", path)
print(submission.head())

In [ ]:
# 見本の gender_submission.csv と同じ予測になっているか
sample = pd.read_csv(DATA_DIR / "gender_submission.csv")
print("見本と一致:", submission.equals(sample))